# Deep Learning Baseline (Member 3)
**Research Objective:** Establish the deep learning accuracy ceiling using an Encoder-only Transformer model before testing the generative Knowledge Distillation approach.
**Model:** `roberta-base` (Significantly outperforms standard BERT for support ticket semantics).
**Targets:** `category`, `priority`, `sentiment`
**Approach:** 
- Dataset: Loads the pre-split, locked `train_split.csv` and `test_split.csv`.
- Feature Engineering: Tokenize concatenated `product` + `issue_description`.
- Hyperparameter Tuning: Native HuggingFace `Trainer` integration with `optuna`.
- Evaluation: Custom classification report identically matching the Traditional ML notebook.

In [ ]:
# 1. Install Required Libraries for Colab
!pip install -q transformers datasets optuna evaluate scikit-learn pandas numpy torch accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
import optuna
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import evaluate
import warnings
warnings.filterwarnings('ignore')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 2. Load the Pre-Split Datasets
# Ensure 'train_split.csv' and 'test_split.csv' are uploaded to Colab
df_train = pd.read_csv('train_split.csv')
df_test = pd.read_csv('test_split.csv')

# Feature Engineering: Concatenate context just like the Traditional ML baseline
df_train['text'] = df_train['product'].astype(str) + " - " + df_train['issue_description'].astype(str)
df_test['text'] = df_test['product'].astype(str) + " - " + df_test['issue_description'].astype(str)

targets = ['category', 'priority', 'sentiment']
label_maps = {}
id_maps = {}

# Create Label Encodings for each target
for target in targets:
    unique_labels = df_train[target].unique().tolist()
    label_maps[target] = {label: idx for idx, label in enumerate(unique_labels)}
    id_maps[target] = {idx: label for label, idx in label_maps[target].items()}
    
    df_train[f'{target}_label'] = df_train[target].map(label_maps[target])
    df_test[f'{target}_label'] = df_test[target].map(label_maps[target])

In [ ]:
# 3. Tokenization & Dataset Conversion
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

# Convert to HuggingFace Datasets
hf_train = Dataset.from_pandas(df_train)
hf_test = Dataset.from_pandas(df_test)

hf_train = hf_train.map(tokenize_function, batched=True)
hf_test = hf_test.map(tokenize_function, batched=True)


In [ ]:
# 4. Metrics Definition for the Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    return {"macro_f1": macro_f1}

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1)
    }

In [ ]:
# 5. Optuna Hyperparameter Search & Training Loop
best_models = {}

for target in targets:
    print("="*50)
    print(f"--- TUNING AND TRAINING FOR: {target.upper()} ---")
    print("="*50)
    
    num_labels = len(label_maps[target])
    
    # We need to format the dataset to point to the correct label column for this target
    train_dataset = hf_train.rename_column(f'{target}_label', 'labels')
    train_dataset = train_dataset.select_columns(['input_ids', 'attention_mask', 'labels'])
    
    test_dataset = hf_test.rename_column(f'{target}_label', 'labels')
    test_dataset = test_dataset.select_columns(['input_ids', 'attention_mask', 'labels'])
    
    def model_init(trial=None):
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, 
            num_labels=num_labels,
            id2label=id_maps[target],
            label2id=label_maps[target]
        )
    
    training_args = TrainingArguments(
        output_dir=f"./results_{target}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=3, # Base epochs for tuning
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        disable_tqdm=True # Keeps Colab logs clean during tuning
    )
    
    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset, # Using test split as eval for baseline simplification
        compute_metrics=compute_metrics,
    )
    
    # Run Optuna Search (Warning: Set n_trials=3 for speed, 10 for massive accuracy)
    best_trial = trainer.hyperparameter_search(
        direction="maximize",
        backend="optuna",
        hp_space=optuna_hp_space,
        n_trials=3 # <--- CHANGE THIS TO 10 for final paper training overnight
    )
    
    print(f"\nBest Hyperparameters for {target}: {best_trial.hyperparameters}\n")
    
    # Train final model with best params
    for n, v in best_trial.hyperparameters.items():
        setattr(trainer.args, n, v)
        
    trainer.train()
    best_models[target] = trainer


In [ ]:
# 6. Custom Evaluation Output (Matches Traditional ML Exactly)

def print_custom_report(y_true, y_pred, target_name):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    macro_prec = precision_score(y_true, y_pred, average='macro')
    macro_rec = recall_score(y_true, y_pred, average='macro')
    
    print("================================================================================")
    print(f"*** {target_name.upper()}")
    print("================================================================================")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Macro Precision: {macro_prec:.4f}")
    print(f"Macro Recall: {macro_rec:.4f}")
    print("")
    
    # Map IDs back to string labels for the classification report
    y_true_str = [id_maps[target_name][val] for val in y_true]
    y_pred_str = [id_maps[target_name][val] for val in y_pred]
    
    rep = classification_report(y_true_str, y_pred_str, digits=2)
    print(rep)
    print("\n")

for target in targets:
    test_dataset = hf_test.rename_column(f'{target}_label', 'labels').select_columns(['input_ids', 'attention_mask', 'labels'])
    
    # Get predictions
    predictions = best_models[target].predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=-1)
    
    print_custom_report(df_test[f'{target}_label'].tolist(), preds, target)

## ⚙️ Hyperparameter Optimization Checklist (How to tweak this further)
If you want to push the baseline accuracy to its absolute ceiling in Colab, change these values:

1. **`n_trials=3`** -> Increase this to `10` or `20` in the `hyperparameter_search` block. **WARNING:** Training Transformer models is computationally heavy. Tuning 3 targets with 10 trials each could take 4-8 hours on a standard Colab T4 GPU. Run this overnight for your final paper numbers.
2. **`MODEL_NAME`** -> We are using `roberta-base`. If you are running out of RAM, downgrade it to `distilroberta-base`. If you have Colab Pro and want maximum accuracy, upgrade it to `roberta-large` or `microsoft/deberta-v3-base`.
3. **`num_train_epochs=3`** -> Increase to `5` in the `TrainingArguments`. This gives the model more passes over the data to learn the complex minority classes (Urgent priority, Positive sentiment).
